correction tp titanic database


In [14]:
import sqlite3, pandas as pd, urllib.request

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
urllib.request.urlretrieve(url, "train.csv")

conn = sqlite3.connect(":memory:")
pd.read_csv("train.csv").to_sql("titanic", conn, index=False)

def q(titre, sql):
    print(f"\n{titre}\n" + "=" * 60)
    display(pd.read_sql_query(sql, conn))

In [2]:
q("1. Top 5 passagers les plus âgés ayant survécu", """
SELECT Name, Age, Sex, Pclass
FROM titanic
WHERE Survived = 1 AND Age IS NOT NULL
ORDER BY Age DESC
LIMIT 5
""")


1. Top 5 passagers les plus âgés ayant survécu


,Name,Age,Sex,Pclass
0,"Barkworth, Mr. Algernon Henry Wilson",80.0,male,1
1,"Andrews, Miss. Kornelia Theodosia",63.0,female,1
2,"Turkula, Mrs. (Hedwig)",63.0,female,3
3,"Harris, Mr. George",62.0,male,2
4,"Stone, Mrs. George Nelson (Martha Evelyn)",62.0,female,1


In [3]:
q("2. Tarif moyen par classe de voyage", """
SELECT Pclass AS classe,
       COUNT(*) AS nb_passagers,
       ROUND(AVG(Fare), 2) AS tarif_moyen
FROM titanic
GROUP BY Pclass
ORDER BY Pclass
""")


2. Tarif moyen par classe de voyage


,classe,nb_passagers,tarif_moyen
0,1,216,84.15
1,2,184,20.66
2,3,491,13.68


In [4]:
q("3. Taux de survie par sexe", """
SELECT Sex,
       COUNT(*) AS nb_passagers,
       SUM(Survived) AS survivants,
       ROUND(100.0 * AVG(Survived), 2) AS taux_survie_pct
FROM titanic
GROUP BY Sex
""")


3. Taux de survie par sexe


,Sex,nb_passagers,survivants,taux_survie_pct
0,female,314,233,74.20
1,male,577,109,18.89


In [5]:
q("4. Passagers de première classe embarqués à Southampton", """
SELECT PassengerId, Name, Sex, Age, Fare
FROM titanic
WHERE Pclass = 1 AND Embarked = 'S'
ORDER BY Name
""")


4. Passagers de première classe embarqués à Southampton


,PassengerId,Name,Sex,Age,Fare
0,731,"Allen, Miss. Elisabeth Walton",female,29.00,211.3375
1,306,"Allison, Master. Hudson Trevor",male,0.92,151.5500
2,298,"Allison, Miss. Helen Loraine",female,2.00,151.5500
3,499,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.00,151.5500
4,461,"Anderson, Mr. Harry",male,48.00,26.5500
...,...,...,...,...,...
122,319,"Wick, Miss. Mary Natalie",female,31.00,164.8667
123,857,"Wick, Mrs. George Dennick (Mary Hitchcock)",female,45.00,164.8667
124,352,"Williams-Lambert, Mr. Fletcher Fellows",male,NaN,35.0000
125,56,"Woolner, Mr. Hugh",male,NaN,35.5000


In [6]:
q("4b. Nombre de passagers de 1re classe de Southampton", """
SELECT COUNT(*) AS nb_1re_classe_southampton
FROM titanic
WHERE Pclass = 1 AND Embarked = 'S'
""")


4b. Nombre de passagers de 1re classe de Southampton


,nb_1re_classe_southampton
0,127


In [7]:
q("5. Statistiques d'âge par classe", """
SELECT Pclass AS classe,
       COUNT(Age) AS ages_connus,
       MIN(Age) AS age_min,
       MAX(Age) AS age_max,
       ROUND(AVG(Age), 2) AS age_moyen
FROM titanic
GROUP BY Pclass
ORDER BY Pclass
""")


5. Statistiques d'âge par classe


,classe,ages_connus,age_min,age_max,age_moyen
0,1,186,0.92,80.0,38.23
1,2,173,0.67,70.0,29.88
2,3,355,0.42,74.0,25.14


In [8]:
q("6. Passagers avec famille à bord (SibSp + Parch > 0)", """
SELECT COUNT(*) AS avec_famille,
       ROUND(100.0 * AVG(Survived), 2) AS taux_survie_pct
FROM titanic
WHERE SibSp + Parch > 0
""")


6. Passagers avec famille à bord (SibSp + Parch > 0)


,avec_famille,taux_survie_pct
0,354,50.56


In [9]:
q("6b. Détail des passagers avec famille (10 plus grandes familles)", """
SELECT Name, SibSp, Parch, SibSp + Parch AS taille_famille, Survived
FROM titanic
WHERE SibSp + Parch > 0
ORDER BY taille_famille DESC, Name
LIMIT 10
""")


6b. Détail des passagers avec famille (10 plus grandes familles)


,Name,SibSp,Parch,taille_famille,Survived
0,"Sage, Master. Thomas Henry",8,2,10,0
1,"Sage, Miss. Constance Gladys",8,2,10,0
2,"Sage, Miss. Dorothy Edith ""Dolly""",8,2,10,0
3,"Sage, Miss. Stella Anna",8,2,10,0
4,"Sage, Mr. Douglas Bullen",8,2,10,0
5,"Sage, Mr. Frederick",8,2,10,0
6,"Sage, Mr. George John Jr",8,2,10,0
7,"Goodwin, Master. Harold Victor",5,2,7,0
8,"Goodwin, Master. Sidney Leonard",5,2,7,0
9,"Goodwin, Master. William Frederick",5,2,7,0


In [10]:
q("7. Enfants de moins de 10 ans", """
SELECT COUNT(*) AS nb_enfants,
       SUM(Survived) AS survivants,
       ROUND(100.0 * AVG(Survived), 2) AS taux_survie_pct
FROM titanic
WHERE Age < 10
""")


7. Enfants de moins de 10 ans


,nb_enfants,survivants,taux_survie_pct
0,62,38,61.29


In [11]:
q("8. Taux de survie par sexe + classe", """
SELECT Sex, Pclass AS classe,
       COUNT(*) AS nb_passagers,
       SUM(Survived) AS survivants,
       ROUND(100.0 * AVG(Survived), 2) AS taux_survie_pct
FROM titanic
GROUP BY Sex, Pclass
ORDER BY Sex, Pclass
""")


8. Taux de survie par sexe + classe


,Sex,classe,nb_passagers,survivants,taux_survie_pct
0,female,1,94,91,96.81
1,female,2,76,70,92.11
2,female,3,144,72,50.00
3,male,1,122,45,36.89
4,male,2,108,17,15.74
5,male,3,347,47,13.54


In [12]:
q("9. Passagers sans numéro de cabine", """
SELECT COUNT(*) AS sans_cabine,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM titanic), 2) AS pct_total
FROM titanic
WHERE Cabin IS NULL OR Cabin = ''
""")


9. Passagers sans numéro de cabine


,sans_cabine,pct_total
0,687,77.1


In [13]:
q("10. Top 10 des tarifs les plus élevés", """
SELECT Name, Pclass, Fare, Survived
FROM titanic
ORDER BY Fare DESC
LIMIT 10
""")


10. Top 10 des tarifs les plus élevés


,Name,Pclass,Fare,Survived
0,"Ward, Miss. Anna",1,512.3292,1
1,"Cardeza, Mr. Thomas Drake Martinez",1,512.3292,1
2,"Lesurer, Mr. Gustave J",1,512.3292,1
3,"Fortune, Mr. Charles Alexander",1,263.0000,0
4,"Fortune, Miss. Mabel Helen",1,263.0000,1
5,"Fortune, Miss. Alice Elizabeth",1,263.0000,1
6,"Fortune, Mr. Mark",1,263.0000,0
7,"Ryerson, Miss. Emily Borie",1,262.3750,1
8,"Ryerson, Miss. Susan Parker ""Suzette""",1,262.3750,1
9,"Baxter, Mr. Quigg Edmond",1,247.5208,0
